# Hypothetical Document Embeddings (HyDE)

Hypothetical Document Embeddings (HyDE) addresses the vocabulary and stylistic mismatch between brief user queries and rich reference documents. Using an instruction-following Groq LLM (`openai/gpt-oss-120b`), HyDE first generates a hypothetical answer, embeds that hallucinated document using HuggingFace MiniLM, and queries the vector store for actual factual documents inhabiting that semantic neighborhood.

## Workflow Architecture

<div align="center">
  <img src="../images/workflow_hyde.png" alt="Hypothetical Document Embeddings (HyDE) Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. Original Query"]):::startNode
    end
    subgraph Gen["Hypothetical Generation"]
        LLM["2. LLM Generator<br/><b>Groq gpt-oss-120b</b><br/>(Hallucinates Hypothetical Document)"]:::llmNode
        HypoDoc["3. Hypothetical Document<br/>(Captures Answer Structure)"]:::docNode
    end
    subgraph Ret["Semantic Retrieval"]
        Emb["4. Embedding Model<br/><b>HuggingFace MiniLM</b><br/>(Embeds Hypothetical Document)"]:::embNode
        VS["5. FAISS Vector Store<br/>(Cosine Nearest Neighbor Search)"]:::vsNode
        RealDocs["6. Retrieved Ground Truth Documents"]:::realDocNode
    end
    subgraph Out["Final Synthesis"]
        Synth["7. Final LLM Synthesis<br/>(Answers Query with Retrieved Facts)"]:::synthNode
        Ans(["8. Grounded Truth Answer"]):::endNode
    end
    Q --> LLM
    LLM --> HypoDoc
    HypoDoc --> Emb
    Emb --> VS
    VS --> RealDocs
    Q --> Synth
    RealDocs --> Synth
    Synth --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef llmNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef docNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef embNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef vsNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef realDocNode fill:#E8EAF6,stroke:#3F51B5,stroke-width:2px,color:#1A237E;
    classDef synthNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Hypothetical Answer Formulation**: Synthesizes speculative text capturing the linguistic structure of the true target passage.
- **Semantic Neighborhood Search**: HuggingFace MiniLM embeds the hypothetical text to locate authentic matching passages.
- **Zero-Shot Relevance Boosting**: 显著 increases retrieval precision on questions where standard query keywords are sparse.


In [1]:
import datetime
import wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Configure Wikipedia API user-agent and rate limiting to prevent HTTP 429 / JSONDecodeError
wikipedia.set_user_agent("MyHydeRagApp/1.0 (contact: student@example.com)")
wikipedia.set_rate_limiting(True, min_wait=datetime.timedelta(seconds=1))

chunk_size = 300
chunk_overlap = 100
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)


C:\Users\itsar\AppData\Local\Temp\ipykernel_15336\753855010.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader
c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

llm = init_chat_model(model="groq:openai/gpt-oss-120b")

db = Chroma.from_documents(documents=docs, embedding=embedding, persist_directory="output/steve_jobs_for_hyde.db")
base_retriver = db.as_retriever(search_kwargs={'k': 5})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5862.58it/s]


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Imagine you are an expert writing a detailed explanation on the topic: '{query}'. Create a hypothetical answer for the topic.")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

def get_hyde_doc(query: str) -> str:
    return hyde_chain.invoke({"query": query})


In [4]:
query = "When was Steve Jobs fired from Apple?"
matched_doc = base_retriver.invoke(get_hyde_doc(query))
print(matched_doc)

[Document(metadata={'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell Wozniak\'s Apple I personal co

In [ ]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_community.document_loaders import TextLoader


loader = TextLoader(os.path.join("..", "data", "langchain_crewai_dataset.txt") if os.path.exists(os.path.join("..", "data", "langchain_crewai_dataset.txt")) else os.path.join("data", "langchain_crewai_dataset.txt"))
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
""" 
HypotheticalDocumentEmbedder is LangChain’s built-in implementation of HyDE (Hypothetical Document Embeddings), based on the 2022 research paper "Precise Zero-Shot Dense Retrieval without Relevance Labels".

It is an embedding class that wraps an LLM and an embedding model together to solve the fundamental query-document asymmetry problem in vector search.

The Problem It Solves
In standard vector search, there is a mismatch:

User Query: Usually short, interrogative, and lacks domain vocabulary (e.g., "How does backprop work?").
Target Documents: Long, explanatory, formal paragraphs with technical terms.
Because of this difference in length, tone, and grammar, their vector embeddings might not sit close together in vector space.
"""


# Step 3: HyDE Embedder using a built-in prompt template
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(llm=llm,base_embeddings=base_embeddings,prompt_key="web_search")

# or Custom HyDE Prompt
# custom = PromptTemplate.from_template(
# "Generate a concise hypothetical answer for this topic: {query}"
# )
# hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
# llm=llm,
# base_embeddings=base_embeddings,
# custom_prompt=custom
# )

vectorstore = Chroma.from_documents(documents=chunks,embedding=hyde_embedding_function,persist_directory="output/langchain")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10457.84it/s]


In [6]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
Context:
{context}
Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    response = rag_chain.invoke({"input": query, "context": matched_docs})
    return response

query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("Final Answer:\n", answer)

Final Answer:
 LangChain includes memory modules such as **ConversationBufferMemory** and **ConversationSummaryMemory**. These tools let the LLM keep track of earlier dialogue turns (buffer) or create concise summaries of longer interactions so they stay within token limits.


Custom HyDE Prompt

In [8]:
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)

# Create vector store with custom HyDE embedding function
vectorstore_custom = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function
)

query = "What memory modules does LangChain provide?"
matched_docs = vectorstore_custom.similarity_search(query, k=4)
response = rag_chain.invoke({"input": query, "context": matched_docs})
print("Final Answer:\n", response)


Final Answer:
 LangChain includes memory modules such as **ConversationBufferMemory** and **ConversationSummaryMemory**. These let the LLM keep track of prior dialogue turns (buffer) or create concise summaries of longer conversations to stay within token limits.
